# GIF Transition Pipeline
Generate numbered transition gifs, review them, then display all together.

## Setup

## Config

In [ ]:
from PIL import Image
import os
import torch
import numpy as np
from diffusers import AutoPipelineForImage2Image
from diffusers.utils import load_image
from IPython.display import display, HTML, Image as IPyImage
import base64

device = "cuda"
print(f"Using device: {device}")    

# --- Paths ---
IMAGES_DIR = r"C:\Users\Calder S. Anderson\Desktop\PARSONS SENIOR YEAR\STUDIO\stabilityaiturboxd_pipeline\Calder\images"          # folder of anchor images (sorted alphabetically)
GIFS_DIR   = r"C:\Users\Calder S. Anderson\Desktop\PARSONS SENIOR YEAR\STUDIO\stabilityaiturboxd_pipeline\Calder\outputs\gifs"    # where numbered gifs are saved
os.makedirs(GIFS_DIR, exist_ok=True)

M          = 30       # intermediate frames per transition
NUM_STEPS  = 4        # diffusion inference steps (SDXL-Turbo: 1-4)
GUIDANCE   = 0.0      # guidance_scale (must stay 0.0 for SDXL-Turbo)
DURATION   = 60       # ms per gif frame
SEED       = 1234     # fixed seed -> less flicker
SIZE       = (1024, 1024)  # SDXL-Turbo native resolution
PROMPT     = "digital photo, natural daylight, amateur photography, sharp, realistic, unedited, casual documentation"


Using device: cuda


## Load Model

In [ ]:
pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype = torch.float16,
    variant     = "fp16",
).to(device)
pipe.safety_checker = None
print("Model loaded.")

Loading pipeline components...: 100%|██████████| 5/5 [00:01<00:00,  3.01it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_img2img.StableDiffusionImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Model loaded.


## Core Function: Generate n.gif

Generates a single transition gif from `src` to `tgt` and saves it as `{n}.gif` in `GIFS_DIR`.

The chain: pass the last frame of the previous gif as `src` so drift carries forward naturally.

In [ ]:
def generate_gif(n, src, tgt):
    """
    Generate transition gif from src -> tgt, saved as {n}.gif in GIFS_DIR.

    Parameters
    ----------
    n   : int         â€” output filename (e.g. 3 -> '3.gif')
    src : PIL.Image   â€” starting frame (pass last frame of previous gif to chain)
    tgt : PIL.Image   â€” ending anchor image

    Returns
    -------
    frames : list of PIL.Image   â€” all frames including src and tgt
    """
    gen    = torch.Generator(device=device).manual_seed(SEED)
    frames = [src]
    prev   = src

    for k in range(1, M + 1):
        # lam      = k / (M + 1)
        # beta     = 0.85 * (lam ** 2)
        # init     = Image.blend(prev, tgt, beta)
        # strength = 0.35 + 0.35 * (lam ** 1.7)
        lam = k / (M + 1)   # 0 â†’ 1 (excluding endpoints)
        beta = 0.85 * (lam ** 2.5) # used to be 2
        init = Image.blend(prev, tgt, beta)
        # strength = 0.55 + 0.40 * lam   # tops out ~0.95
        strength = 0.35 + 0.25 * lam   # tops out ~0.95


        img = pipe(
            prompt              = PROMPT,
            image               = init,
            strength            = float(strength),
            guidance_scale      = float(GUIDANCE),
            num_inference_steps = int(NUM_STEPS),
            generator           = gen,
        ).images[0]
        # color correction
        from PIL import ImageEnhance
        img = ImageEnhance.Color(img).enhance(0.95)
        r, g, b = img.split()
        b = b.point(lambda i: i * 1.02)
        r = r.point(lambda i: i * 0.98)
        img = Image.merge("RGB", (r, g, b))

        frames.append(img)
        prev = img

    frames.append(tgt)




    gif_path = os.path.join(GIFS_DIR, f"{n}.gif")
    frames[0].save(
        gif_path,
        save_all      = True,
        append_images = frames[1:],
        duration      = DURATION,
        loop          = 0,
    )
    print(f"Saved {gif_path}  ({len(frames)} frames)")
    return frames

## Run: Generate All Gifs (Chained)

Reads all images from `IMAGES_DIR` in sorted order and generates one gif per consecutive pair.
The last frame of each gif becomes the `src` of the next â€” edit and re-run individual cells below to redo specific gifs.

In [ ]:
# Load anchor images sorted by filename
supported = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
image_files = sorted(
    f for f in os.listdir(IMAGES_DIR)
    if os.path.splitext(f)[1].lower() in supported
)
anchors = [
    Image.open(os.path.join(IMAGES_DIR, f)).convert("RGB").resize(SIZE)
    for f in image_files
]
print(f"Loaded {len(anchors)} anchor images: {image_files}")

Loaded 123 anchor images: ['000.jpg', '000.webp', '001.jpg', '001.webp', '002.jpg', '002.webp', '003.jpg', '003.png', '004.jpg', '005.webp', '006.webp', '007.jpg', '008.jpg', '009.jpg', '0x0.webp', '15936576199327742.jpg', '15936576199327743.webp', '15936576199327744.webp', '15936576199327745.webp', '15936576199327746.webp', '15936576199327747.webp', '15936576199327748.webp', '15936576199327749.webp', '15936576199327750.webp', '15936576199327751.webp', '15936576199327752.webp', '15936576199327753.webp', '15936576199327754.webp', '15936576199327755.webp', '15936576199327756.webp', '15936576199327757.webp', '15936576199327758.webp', '15936576199327759.webp', '15936576199327760.webp', '15936576199327761.webp', '15936576199327762.webp', '15936576199327763.webp', '15936576199327764.webp', '15936576199327765.webp', '15936576199327766.jpg', '15936576199327767.jpg', '1617792549838.jpg', '17_image-png.png', '1W8x6ZJh2fAZG.webp', '1_vetvGc761MUC3_NuFyJYWg.jpg', '2013_alma_0002-scaled.webp', '201

In [ ]:
# Generate all gifs in chained order
# last frame of gif N becomes src of gif N+1; wraps back to anchors[0] at the end
all_frames = {}
chain_src  = anchors[0]

for i, tgt in enumerate(anchors[1:] + [anchors[0]], start=1):
    print(f"Generating {i}.gif ...")
    frames = generate_gif(n=i, src=chain_src, tgt=tgt)
    all_frames[i] = frames
    chain_src = frames[-1]   # hand last frame forward

print("\nAll gifs generated.")

Generating 1.gif ...


100%|██████████| 4/4 [00:01<00:00,  3.66it/s]


Saved outputs/gifs\1.gif  (32 frames)
Generating 2.gif ...


100%|██████████| 4/4 [00:00<00:00,  4.41it/s]


Saved outputs/gifs\2.gif  (32 frames)
Generating 3.gif ...


100%|██████████| 4/4 [00:01<00:00,  2.08it/s]


Saved outputs/gifs\3.gif  (32 frames)
Generating 4.gif ...


100%|██████████| 4/4 [00:01<00:00,  2.07it/s]


Saved outputs/gifs\4.gif  (32 frames)
Generating 5.gif ...


100%|██████████| 3/3 [00:01<00:00,  1.93it/s]


In [ ]:
import math
import matplotlib.pyplot as plt

def show_gif_frames_table(all_frames, max_cols=None, figsize_per_cell=(2.2, 2.2), title=True):
    """
    Display frames of each gif (stored in all_frames dict) in a big grid.
    
    all_frames: dict[int, list[PIL.Image]]
        e.g. {1: [img0, img1, ...], 2: [...], ...}
    max_cols: int | None
        If set, cap the number of columns shown (useful if there are lots of frames).
    figsize_per_cell: (w, h)
        Size per cell in inches.
    """
    gif_ids = sorted(all_frames.keys())
    n_gifs = len(gif_ids)
    if n_gifs == 0:
        print("all_frames is empty.")
        return

    # Determine number of columns = max frames across gifs
    max_frames = max(len(all_frames[i]) for i in gif_ids)
    n_cols = min(max_frames, max_cols) if max_cols is not None else max_frames

    # Figure size scales with grid
    fig_w = max(10, n_cols * figsize_per_cell[0])
    fig_h = max(4, n_gifs * figsize_per_cell[1])

    fig, axes = plt.subplots(n_gifs, n_cols, figsize=(fig_w, fig_h))

    # Normalize axes into 2D array indexing [row, col]
    if n_gifs == 1 and n_cols == 1:
        axes = [[axes]]
    elif n_gifs == 1:
        axes = [axes]  # shape -> [1][n_cols]
    elif n_cols == 1:
        axes = [[ax] for ax in axes]  # shape -> [n_gifs][1]

    for r, gif_id in enumerate(gif_ids):
        frames = all_frames[gif_id]
        for c in range(n_cols):
            ax = axes[r][c]
            ax.axis("off")

            if c < len(frames):
                ax.imshow(frames[c])
                ax.set_title(f"gif {gif_id} Â· f{c}", fontsize=8)
            else:
                # blank cell
                ax.set_facecolor("black")

    if title:
        fig.suptitle("All GIFs: frames laid out by (gif row, frame column)", fontsize=14)
        plt.subplots_adjust(top=0.92)

    plt.tight_layout()
    plt.show()


# Use it:
show_gif_frames_table(all_frames, max_cols=None)

## Re-run a Single Gif

If a gif looks bad, re-run just that one. Load the last frame of the previous gif manually to stay in chain.

In [ ]:
# Example: redo gif 3
# Uncomment and adjust as needed

# n = 3
# prev_gif = Image.open(os.path.join(GIFS_DIR, f"{n-1}.gif"))
# prev_gif.seek(prev_gif.n_frames - 1)
# chain_src = prev_gif.copy().convert("RGB")
# tgt = anchors[n % len(anchors)]   # wraps for the last gif
# frames = generate_gif(n=n, src=chain_src, tgt=tgt)
# all_frames[n] = frames

## Display All Gifs in Folder

Reads every `.gif` from `GIFS_DIR` in numeric order and displays them in a scrollable inline grid.
Delete any bad gifs from the folder and re-run this cell to review what remains.

In [ ]:
def combine_gifs(gifs_dir=GIFS_DIR, output_path="outputs/master_5.gif", duration=DURATION):
    """
    Combine all .gif files in gifs_dir into one master gif.

    Parameters
    ----------
    gifs_dir    : str â€” folder containing numbered .gif files
    output_path : str â€” where to save the combined gif
    duration    : int â€” ms per frame in the output gif

    Returns
    -------
    output_path : str â€” path to the saved master gif
    """
    gif_files = sorted(
        [f for f in os.listdir(gifs_dir) if f.endswith(".gif")],
        key=lambda x: int(os.path.splitext(x)[0]) if os.path.splitext(x)[0].isdigit() else 0
    )

    if not gif_files:
        print(f"No gifs found in {gifs_dir!r}")
        return None

    print(f"Combining {len(gif_files)} gifs into {output_path!r} ...")

    all_frames = []

    for i, fname in enumerate(gif_files):
        path = os.path.join(gifs_dir, fname)
        gif  = Image.open(path)

        # Extract all frames from this gif
        frames = []
        try:
            while True:
                frames.append(gif.copy().convert("RGB"))
                gif.seek(gif.tell() + 1)
        except EOFError:
            pass

        # For intermediate gifs, skip first and last frame to avoid duplicates
        # (first frame = last frame of previous gif, last frame = first frame of next gif)
        if i == 0:
            # First gif: keep all frames except the last (it's duplicated in gif 2)
            all_frames.extend(frames[:-1])
        elif i == len(gif_files) - 1:
            # Last gif: skip first frame (duplicated from previous), keep the rest
            all_frames.extend(frames[1:])
        else:
            # Middle gifs: skip both first and last
            all_frames.extend(frames[1:-1])

        print(f"  {fname}: {len(frames)} frames ({len(frames) if i == 0 else len(frames) - 1 if i == len(gif_files) - 1 else len(frames) - 2} kept)")

    # Save combined gif
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    all_frames[0].save(
        output_path,
        save_all      = True,
        append_images = all_frames[1:],
        duration      = duration,
        loop          = 0,
    )

    print(f"\nSaved master gif with {len(all_frames)} total frames to {output_path!r}")
    return output_path


# Run it
master_path = combine_gifs()

# Display the result
if master_path and os.path.exists(master_path):
    display(IPyImage(filename=master_path))

# 22.4s 